# # M06 - Transformers from the inside
#
# **Hardware:** CPU (about 30 seconds of training) · **Time:** 3-4 hours ·
# **Prerequisite:** M04-M05, basic PyTorch
#
# This notebook builds a transformer language model from scratch and trains it on
# a small synthetic corpus. Nothing is downloaded and no API key is needed. By
# the end you will have measured the effect of the tokenizer, the causal mask,
# the parameter budget, the learning-rate schedule, and the decoding temperature
# on the same model.
#
# The model is small on purpose: two layers, 96 embedding dimensions, about
# 240,000 parameters. It is large enough to learn the grammar of the corpus and
# small enough that every number below is reproducible on a laptop.
#
# **You will produce:** a trained checkpoint (`outputs/m06_tiny_gpt.pt`), a
# tokenizer comparison, a loss curve, and generation samples at three
# temperatures.

# ## 1. Setup and data
#
# The corpus is generated by `scripts/make_training_data.py` from a fixed seed.
# It contains 2,200 synthetic sentences built from a small grammar: eight
# subjects, six places, seven objects, seven verbs, six endings. That structure
# is deliberate. A model can learn it from a few hundred steps, and because the
# vocabulary is small we can read every generated token.

In [1]:
from pathlib import Path
import math
import sys

import numpy as np
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "projects").is_dir())
sys.path.insert(0, str(ROOT / "projects" / "llm-fine-tuning"))

import llm_core as lc

print("torch", torch.__version__, "| device", "cuda" if torch.cuda.is_available() else "cpu")
print("repo root", ROOT.name)
assert sys.version_info >= (3, 10)

lc.ensure_datasets(quiet=True)
corpus = lc.load_corpus()
sentences = lc.corpus_sentences()
stats = lc.vocab_preview()
print({key: value for key, value in stats.items() if key != "most_common_words"})
print("first three sentences:")
for sentence in sentences[:3]:
    print("  ", sentence)
assert len(sentences) > 500 and len(corpus) > 100_000

torch 2.7.1+cpu | device cpu
repo root AITopics
{'characters': 170828, 'lines': 2566, 'words': 32834, 'unique_words': 66, 'unique_characters': 28}
first three sentences:
   The girl offered a broken wheel at the edge of the forest before the sun went down.
   The girl remembered a loaf of bread in the quiet village and said nothing.
   The old boatman carried a folded map near the river and smiled at the sky.


# ## 2. Three tokenizers, one trade-off
#
# A tokenizer turns text into integers. The choice is a trade-off between
# **vocabulary size** (parameters in the embedding table) and **sequence length**
# (compute and context used per sentence).
#
# * Character tokenizer: 30 tokens for this corpus, ~90 tokens per sentence.
# * Word tokenizer: larger vocabulary, ~15 tokens per sentence.
# * Byte-pair encoding: starts from 256 bytes and merges frequent pairs, so it
#   keeps common words whole without an unbounded vocabulary.
#
# Production models use BPE (or a close relative). The exercise at the end asks
# you to compute the embedding-parameter cost of each choice.

In [2]:
char_tokenizer = lc.CharTokenizer.from_text(corpus)
word_tokenizer = lc.WordTokenizer.from_text(corpus, lowercase=False)
sample = "\n".join(sentences[:60])
bpe_tokenizer = lc.BPETokenizer.train(sample, vocab_size=280)
print("BPE merges learned:", len(bpe_tokenizer.merges))

comparison_text = "\n".join(sentences[:12])
rows = lc.tokenizer_comparison(
    {"char": char_tokenizer, "word": word_tokenizer, "bpe": bpe_tokenizer}, comparison_text
)
print(lc.format_table(rows, ["name", "vocab_size", "tokens", "characters", "chars_per_token"]))

for tokenizer in (char_tokenizer, word_tokenizer, bpe_tokenizer):
    probe = "The fox found a silver key near the river."
    assert tokenizer.decode(tokenizer.encode(probe)) == probe, type(tokenizer).__name__
print("round-trip ok for all three tokenizers")

BPE merges learned: 24
name  vocab_size  tokens  characters  chars_per_token
----  ----------  ------  ----------  ---------------
char  30          938     938         1.0            
word  68          190     938         4.94           
bpe   280         581     938         1.61           
round-trip ok for all three tokenizers


# Look at what BPE actually learned. Early merges join the most frequent byte
# pairs; later merges assemble whole words. The first ten merges usually look
# like fragments, which is exactly why subword tokenizers handle unseen words
# without an `<unk>` explosion.

In [3]:
def render(ids):
    return bpe_tokenizer.decode(ids)


seen = {}
for index, (left, right) in enumerate(bpe_tokenizer.merges[:10]):
    seen[(left, right)] = index
print("first ten merges:")
for (left, right), index in seen.items():
    print(f"  {index:2d}: {render([left])!r} + {render([right])!r} -> {render([256 + index])!r}")

first ten merges:
   0: 'e' + ' ' -> 'e '
   1: 'h' + 'e ' -> 'he '
   2: 'd' + ' ' -> 'd '
   3: 't' + 'he ' -> 'the '
   4: 'e' + 'r' -> 'er'
   5: ' ' + 'the ' -> ' the '
   6: 'e' + 'd ' -> 'ed '
   7: 'T' + 'he ' -> 'The '
   8: 'a' + ' ' -> 'a '
   9: '\n' + 'The ' -> '\nThe '


# ## 3. From tokens to training batches
#
# A causal language model learns one thing: given tokens `0..t`, predict token
# `t+1`. A batch is therefore two copies of the same stream, shifted by one.
# `get_batch` samples random windows; the printed pair shows that `target` is
# `input` moved one character to the left.
#
# We train on the full character stream rather than sentence by sentence. This is
# what "pre-training on a corpus" means: no labels, no prompt format, just the
# next token.

In [4]:
ids = char_tokenizer.encode(corpus)
tokens = torch.tensor(ids, dtype=torch.long)
print("tokens", len(tokens), "block size 128 ->", len(tokens) // 128, "non-overlapping windows")
x, y = lc.get_batch(tokens, batch_size=2, block_size=24, generator=torch.Generator().manual_seed(0))
print("input :", repr(char_tokenizer.decode(x[0].tolist())))
print("target:", repr(char_tokenizer.decode(y[0].tolist())))
assert torch.equal(x[0, 1:], y[0, :-1])

tokens 170828 block size 128 -> 1334 non-overlapping windows
input : 'the old oak and smiled a'
target: 'he old oak and smiled at'


# ## 4. Attention by hand
#
# Self-attention compares every position with every other position. For a
# sequence of `T` tokens and head dimension `d`, the score matrix is `T x T`,
# scaled by `1/sqrt(d)`, masked so position `t` cannot see `t+1`, and normalised
# with softmax.
#
# The mask is not an optimisation. Remove it and the model can read the answer it
# is supposed to predict: training loss collapses and generation fails, because
# at inference time the future tokens do not exist.

In [5]:
def softmax(matrix):
    shifted = matrix - matrix.max(axis=-1, keepdims=True)
    exponent = np.exp(shifted)
    return exponent / exponent.sum(axis=-1, keepdims=True)


rng = np.random.default_rng(0)
sequence_length, head_dim = 6, 8
q = rng.normal(size=(sequence_length, head_dim))
k = rng.normal(size=(sequence_length, head_dim))
scores = q @ k.T / math.sqrt(head_dim)
causal = np.tril(np.ones((sequence_length, sequence_length), dtype=bool))
masked = np.where(causal, scores, -np.inf)
weights = softmax(masked)

print("attention weights (rows sum to 1):")
for row in weights:
    print("  " + " ".join(f"{value:.2f}" for value in row))
print("row sums:", np.round(weights.sum(axis=1), 6))
print("upper triangle is exactly zero:", bool(np.all(weights[~causal] == 0.0)))
assert np.allclose(weights.sum(axis=1), 1.0)
assert np.all(weights[~causal] == 0.0)

attention weights (rows sum to 1):
  1.00 0.00 0.00 0.00 0.00 0.00
  0.41 0.59 0.00 0.00 0.00 0.00
  0.47 0.12 0.41 0.00 0.00 0.00
  0.60 0.16 0.05 0.19 0.00 0.00
  0.10 0.15 0.20 0.30 0.25 0.00
  0.02 0.05 0.24 0.12 0.05 0.52
row sums: [1. 1. 1. 1. 1. 1.]
upper triangle is exactly zero: True


# ## 5. The model and its parameter budget
#
# Every parameter in `TinyGPT` belongs to one of five groups. With `C = n_embd`,
# `V = vocab_size`, `T = block_size` and `L = n_layer`:
#
# | Group | Parameters |
# | --- | --- |
# | token embedding | `V * C` |
# | position embedding | `T * C` |
# | each attention block | `4C^2 + 4C` (QKV, output projection, biases) |
# | each MLP block | `8C^2 + 5C` (expand to 4C, project back) |
# | each pair of LayerNorms | `4C` |
# | final LayerNorm | `2C` |
# | output head | `V * C`, or zero when tied to the token embedding |
#
# The table below computes this from the configuration and checks it against the
# model. This is the arithmetic behind every "7B parameters" claim you will read.

In [6]:
config = dict(block_size=128, n_layer=2, n_head=4, n_embd=96, dropout=0.1)
model = lc.build_model(len(char_tokenizer), seed=0, **config)
C, V, T, L = config["n_embd"], len(char_tokenizer), config["block_size"], config["n_layer"]

parts = {
    "token embedding": V * C,
    "position embedding": T * C,
    "attention blocks": L * (4 * C * C + 4 * C),
    "MLP blocks": L * (8 * C * C + 5 * C),
    "layer norms": L * 2 * (2 * C) + 2 * C,
    "output head (tied)": 0,
}
rows = [{"component": name, "parameters": value} for name, value in parts.items()]
rows.append({"component": "total (computed)", "parameters": sum(parts.values())})
rows.append({"component": "total (model)", "parameters": model.parameter_count()})
print(lc.format_table(rows, ["component", "parameters"]))
print("float32 size:", round(model.param_bytes(4) / 1024, 1), "KiB")
assert sum(parts.values()) == model.parameter_count()

component           parameters
------------------  ----------
token embedding     2880      
position embedding  12288     
attention blocks    74496     
MLP blocks          148416    
layer norms         960       
output head (tied)  0         
total (computed)    239040    
total (model)       239040    
float32 size: 933.8 KiB


# ## 6. Training
#
# The schedule warms the learning rate up over 20 steps, then decays it with a
# cosine curve. Warm-up avoids destroying the random initialisation with a large
# first step; the cosine tail lets the model settle.
#
# Four hundred steps take about 25 seconds on CPU. The loss is cross-entropy in
# nats per character; `exp(loss)` is perplexity, the effective number of equally
# likely next characters under the model.

In [7]:
history = lc.train_lm(
    model,
    ids,
    steps=400,
    batch_size=16,
    block_size=128,
    lr=3e-3,
    seed=0,
    warmup=20,
    log_every=100,
)
summary = history.summary()
print(summary)
print("logged losses:", [round(value, 3) for value in history.losses])
print("perplexity:", round(lc.perplexity(history.losses[-1]), 2))
print("tokens/second:", summary["tokens_per_second"])
assert history.losses[-1] < history.losses[0]
assert history.losses[-1] < 1.0
assert summary["tokens"] == 400 * 16 * 128

{'steps': 400, 'first_loss': 3.4543, 'last_loss': 0.4328, 'seconds': 22.76, 'tokens': 819200, 'tokens_per_second': 36001, 'evals': []}
logged losses: [3.454, 1.618, 0.948, 0.599, 0.433]
perplexity: 1.54
tokens/second: 36001


# ## 7. Generation and temperature
#
# At temperature 0 the model always picks the most likely next character. Higher
# temperatures flatten the distribution; `top_k` removes the tail before
# sampling. The samples are imperfect because the model is tiny - this is the
# honest version of a "language model demo", not a claim of fluency.

In [8]:
def sample(prompt, temperature, top_k=None, seed=1):
    return lc.generate_text(
        model, char_tokenizer, prompt, max_new_tokens=70, temperature=temperature, top_k=top_k, seed=seed
    )

for label, temperature, top_k in [
    ("greedy", 0.0, None),
    ("warm", 0.5, 20),
    ("hot", 1.0, None),
]:
    text = sample("The old boatman", temperature, top_k)
    print(f"\n[{label}] repetition={lc.repetition_rate(text):.2f} distinct-2={lc.distinct_n([text], 2):.2f}")
    print(repr(text))

first = sample("The fox", 0.0)
second = sample("The fox", 0.0)
assert first == second, "greedy decoding must be deterministic"


[greedy] repetition=0.00 distinct-2=1.00
' remembered a silver key the sidge while the wind rose.\nThe baker cat '

[warm] repetition=0.00 distinct-2=1.00
' rema remembered a silver key on the northern road before the sun went'



[hot] repetition=0.00 distinct-2=1.00
' rema ril of the n the rivillagre the the wind rose.\nThe old boatman l'


# ## 8. Does it memorise or generalise?
#
# The corpus has a fixed grammar, so a good model should continue a new sentence
# in the same style. A cheating model would reproduce training sentences
# verbatim. Greedy completion from a training prefix is a crude memorisation
# probe: we ask for 20 characters and count exact matches against the real
# continuation.
#
# A low rate means the model is generating from the grammar rather than copying
# stored text. It does not prove generalisation - the grammar itself is narrow -
# but it rules out the most obvious failure.

In [9]:
exact = 0
probes = sentences[100:110]
for sentence in probes:
    prefix, truth = sentence[:20], sentence[20:40]
    generated = lc.generate_text(model, char_tokenizer, prefix, max_new_tokens=20, temperature=0.0)
    exact += int(generated.startswith(truth))
print(f"verbatim continuation: {exact}/{len(probes)}")

verbatim continuation: 0/10


# ## 9. Save the checkpoint
#
# The checkpoint stores the state dict and the configuration needed to rebuild
# the model. A checkpoint without its tokenizer and configuration is only half an
# artifact: the weights are meaningless if you cannot map ids back to text.

In [10]:
output_dir = ROOT / "notebooks" / "llm-fine-tuning" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "m06_tiny_gpt.pt"
torch.save({"state_dict": model.state_dict(), "config": model.config.as_dict(), "vocab": char_tokenizer.vocab}, checkpoint_path)
print("saved", checkpoint_path.name, round(checkpoint_path.stat().st_size / 1024, 1), "KiB")

loaded = torch.load(checkpoint_path, weights_only=False)
restored = lc.TinyGPT(lc.GPTConfig(**loaded["config"]))
restored.load_state_dict(loaded["state_dict"])
model.eval()
restored.eval()
with torch.no_grad():
    a, _ = model(x[:1])
    b, _ = restored(x[:1])
print("max logit difference after reload:", float((a - b).abs().max()))
assert torch.allclose(a, b, atol=1e-6)

saved m06_tiny_gpt.pt 944.2 KiB
max logit difference after reload: 0.0


# ## 10. Validation and exercises
#
# The cell below asserts the claims made above. If every line prints `ok`, the
# notebook is reproducible from a clean kernel.
#
# **Exercises**
#
# 1. Change `n_embd` to 64 and retrain. Compute the new parameter count with the
#    formula first, then check it against the model. How much slower is training?
# 2. Remove the causal mask in `CausalSelfAttention` (train a copy). Report the
#    loss and explain why it is not a valid language model.
# 3. Train the word tokenizer variant. It has a shorter sequence but a larger
#    embedding table. Which is cheaper at this corpus size, and why?
# 4. Sample at temperature 2.0. Describe what happens to repetition and
#    distinct-2. Explain using the softmax curve.
# 5. Optional GPU path: move `model` and the token tensors to `cuda`, keep
#    `batch_size=16`, and compare tokens/second. Do not change the hyperparameters
#    and claim a quality improvement - only the speed changes.

In [11]:
checks = {
    "corpus loaded": len(corpus) > 100_000,
    "tokenizers round-trip": all(
        tokenizer.decode(tokenizer.encode("The fox found a key.")) == "The fox found a key."
        for tokenizer in (char_tokenizer, word_tokenizer, bpe_tokenizer)
    ),
    "targets are shifted inputs": torch.equal(x[0, 1:], y[0, :-1]),
    "causal mask hides the future": bool(np.all(weights[~causal] == 0.0)),
    "parameter formula matches": sum(parts.values()) == model.parameter_count(),
    "loss decreased": history.losses[-1] < history.losses[0],
    "checkpoint round-trips": torch.allclose(a, b, atol=1e-6),
}
for name, passed in checks.items():
    print(f"{'ok' if passed else 'FAIL'} - {name}")
    assert passed, name
print("\nM06 validation passed.")

ok - corpus loaded
ok - tokenizers round-trip
ok - targets are shifted inputs
ok - causal mask hides the future
ok - parameter formula matches
ok - loss decreased
ok - checkpoint round-trips

M06 validation passed.
